In [ ]:
class Stack:
    def __init__(self):
        self.itens = []

    def is_empty(self):
        return len(self.itens) == 0

    def push(self, item):
        self.itens.append(item)

    def pop(self):
        if self.is_empty():
            return None
        return self.itens.pop()

    def peek(self):
        if self.is_empty():
            return None
        return self.itens[-1]

    def size(self):
        return len(self.itens)

In [ ]:
def decimal_para_hexa(n):
    if n < 0:
        raise ValueError("n deve ser não-negativo")
    if n == 0:
        return "0"
    digitos = "0123456789ABCDEF"
    pilha = Stack()
    while n > 0:
        resto = n % 16
        pilha.push(digitos[resto])
        n //= 16
    hexa = ""
    while not pilha.is_empty():
        hexa += pilha.pop()
    return hexa

print(decimal_para_hexa(26))
print(decimal_para_hexa(255))
print(decimal_para_hexa(0))
print(decimal_para_hexa(4095))

In [ ]:
def balanceia_colchetes(expressao):
    pares = {')': '(', ']': '[', '}': '{'}
    pilha = Stack()
    for caractere in expressao:
        if caractere in '([{':
            pilha.push(caractere)
        elif caractere in ')]}':
            if pilha.is_empty():
                return False
            abertura = pilha.pop()
            if abertura != pares[caractere]:
                return False
    return pilha.is_empty()

print(balanceia_colchetes("({[]})"))
print(balanceia_colchetes("({[}])"))
print(balanceia_colchetes("{(a+b)*[c-d]}"))
print(balanceia_colchetes("((a+b)"))

In [ ]:
import pandas as pd

class HistoricoOperacoes:
    def __init__(self, dataframe):
        self.estado_atual = dataframe.copy()
        self.pilha_estados = Stack()

    def aplicar_filtro(self, condicao):
        self.pilha_estados.push(self.estado_atual.copy())
        self.estado_atual = self.estado_atual.loc[condicao].copy()

    def desfazer(self):
        if self.pilha_estados.is_empty():
            print("Nenhum estado anterior para desfazer.")
            return False
        self.estado_atual = self.pilha_estados.pop()
        return True

    def mostrar(self):
        print(self.estado_atual.to_string(index=False))

df = pd.DataFrame({
    "produto": ["Notebook", "Mouse", "Monitor", "Teclado"],
    "vendas": [3200, 800, 2100, 1200]
})

historico = HistoricoOperacoes(df)
historico.aplicar_filtro(historico.estado_atual["vendas"] >= 1000)

print("Após o filtro:")
historico.mostrar()

historico.desfazer()
print("\nApós desfazer:")
historico.mostrar()

In [ ]:
import re

def avaliar_expressao(expressao):
    precedencia = {'+': 1, '-': 1, '*': 2, '/': 2}
    valores = Stack()
    operadores = Stack()

    tokens = re.findall(r'\d+(?:\.\d+)?|[()+\-*/]', expressao.replace(' ', ''))

    def aplicar_operador():
        operador = operadores.pop()
        b = valores.pop()
        a = valores.pop()
        if a is None or b is None:
            raise ValueError("Expressão inválida")
        if operador == '+':
            valores.push(a + b)
        elif operador == '-':
            valores.push(a - b)
        elif operador == '*':
            valores.push(a * b)
        elif operador == '/':
            if b == 0:
                raise ZeroDivisionError("Divisão por zero")
            valores.push(a / b)

    for token in tokens:
        if token.replace('.', '', 1).isdigit():
            valores.push(float(token) if '.' in token else int(token))
        elif token == '(':
            operadores.push(token)
        elif token == ')':
            while not operadores.is_empty() and operadores.peek() != '(':
                aplicar_operador()
            if operadores.is_empty():
                raise ValueError("Parênteses desbalanceados")
            operadores.pop()
        else:
            while (not operadores.is_empty()
                   and operadores.peek() != '('
                   and precedencia[operadores.peek()] >= precedencia[token]):
                aplicar_operador()
            operadores.push(token)

    while not operadores.is_empty():
        if operadores.peek() == '(':
            raise ValueError("Parênteses desbalanceados")
        aplicar_operador()

    resultado = valores.pop()
    if not valores.is_empty():
        raise ValueError("Expressão inválida")
    return resultado

print(avaliar_expressao("3 + 4 * (2 - 1)"))
print(avaliar_expressao("10 / 2 + 6 * 3"))